# Лабораторная работа №4
## Аналитика как конкурентное преимущество

**Задание 1.** Комплексная система мультиплеерной аналитики «Multiplayer Balance & Matchmaking Insights»: качество матчмейкинга, баланс карт, антифрод.
**Задание 2.** Интерактивный экран «Матрица соревновательного баланса персонажей» (Pick Rate vs Win Rate).

Данные — синтетические логи соревновательного шутера 5v5, сгенерированные скриптом `src/generate_data.py` с заложенными «скрытыми аномалиями» для проверки алгоритмов.

## Задание 1. Шаг 1. Проектирование хранилища и загрузка данных

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
REPORTS = '../reports'

In [2]:
# Схема данных: ID — строки, MMR — целые, метрики эффективности — дробные
matches_df = pd.read_csv('../data/matches.csv', parse_dates=['match_date'], keep_default_na=False,
                         dtype={'match_id': 'string', 'map_id': 'string', 'server_region': 'string',
                                'winning_team': 'string', 'duration_sec': 'int64', 'queue_time_sec': 'float64'})
players_df = pd.read_csv('../data/player_stats.csv',
                         dtype={'match_id': 'string', 'player_id': 'string', 'team_side': 'string',
                                'hero_id': 'string', 'weapon_id': 'string', 'kills': 'int64', 'deaths': 'int64',
                                'assists': 'int64', 'mmr_before': 'int64', 'mmr_after': 'int64', 'ping': 'int64'})
reports_df = pd.read_csv('../data/reports.csv', dtype='string')

for name, df in [('matches_df', matches_df), ('players_df', players_df), ('reports_df', reports_df)]:
    print(f"{name}: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
print()
players_df.info()

matches_df: 3,000 строк × 7 столбцов
players_df: 30,000 строк × 11 столбцов
reports_df: 2,319 строк × 4 столбцов

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   match_id    30000 non-null  string
 1   player_id   30000 non-null  string
 2   team_side   30000 non-null  string
 3   hero_id     30000 non-null  string
 4   weapon_id   30000 non-null  string
 5   kills       30000 non-null  int64 
 6   deaths      30000 non-null  int64 
 7   assists     30000 non-null  int64 
 8   mmr_before  30000 non-null  int64 
 9   mmr_after   30000 non-null  int64 
 10  ping        30000 non-null  int64 
dtypes: int64(6), string(5)
memory usage: 2.5 MB


In [3]:
display(matches_df.head(3))
display(players_df.head(3))
display(reports_df.head(3))

# Проверки целостности реляционной структуры
assert matches_df['match_id'].is_unique
assert players_df.groupby('match_id').size().eq(10).all(), 'в каждом матче должно быть 10 игроков'
assert players_df.groupby(['match_id', 'team_side']).size().eq(5).all(), 'команды 5 на 5'
assert players_df['match_id'].isin(matches_df['match_id']).all()
assert reports_df['match_id'].isin(matches_df['match_id']).all()
print("Целостность: 10 игроков на матч, команды 5v5, все ссылки match_id корректны")
print(f"Уникальных игроков: {players_df['player_id'].nunique():,}; "
      f"период: {matches_df['match_date'].min().date()} — {matches_df['match_date'].max().date()}")
print("Жалобы по причинам:", reports_df['reason'].value_counts().to_dict())

,match_id,map_id,duration_sec,server_region,winning_team,queue_time_sec,match_date
0,MATCH_00000,De_Dust2,1375,EU,Defense,112.5,2026-09-06 11:37:00
1,MATCH_00001,De_Inferno,1702,EU,Attack,132.2,2026-09-12 00:49:00
2,MATCH_00002,De_Mirage,1671,ASIA,Defense,75.2,2026-09-06 17:00:00


,match_id,player_id,team_side,hero_id,weapon_id,kills,deaths,assists,mmr_before,mmr_after,ping
0,MATCH_00000,PLAYER_00599,Attack,Hero_9,SMG_MP9,23,16,11,2546,2518,77
1,MATCH_00000,PLAYER_00525,Attack,Hero_8,SMG_MP9,13,19,5,2334,2308,29
2,MATCH_00000,PLAYER_01570,Attack,Hero_20,Shotgun_Nova,7,21,4,2096,2070,72


,match_id,reporter_id,reported_id,reason
0,MATCH_00000,PLAYER_01923,PLAYER_00074,cheating
1,MATCH_00000,PLAYER_00529,PLAYER_00074,cheating
2,MATCH_00000,PLAYER_01384,PLAYER_00599,toxicity


Целостность: 10 игроков на матч, команды 5v5, все ссылки match_id корректны
Уникальных игроков: 1,953; период: 2026-09-01 — 2026-09-14
Жалобы по причинам: {'cheating': 1294, 'toxicity': 859, 'afk': 166}
